# Network Comparison: `small` vs `small_parthenope`

This notebook compares the primordial abundances and time evolution predicted by the `small` network and the `small_parthenope` network.

In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Ensure the repo root is in path
sys.path.append(os.getcwd())

from primat.main import PRIMAT

# Configuration
params = {
    "output_time_evolution": True,
    "verbose": False
}

# Solve for both networks
print("Solving small network...")
run_small = PRIMAT({**params, "network": "small"})
res_small = run_small.solve()

print("Solving small_parthenope network...")
run_parthenope = PRIMAT({**params, "network": "small_parthenope"})
res_parthenope = run_parthenope.solve()

print("Done.")

Solving small network...
[output] Time-evolution data (500 rows) written to /Users/pitrou/Cosmologie/SDrive/iap/PyPRIMAT/results/output_tables.tsv
Solving small_parthenope network...
[output] Time-evolution data (500 rows) written to /Users/pitrou/Cosmologie/SDrive/iap/PyPRIMAT/results/output_tables.tsv
Done.


In [ ]:
# Compare final abundances using Y_final (no _t_vec needed)
nuclides = [n for n in run_small.abundance_names
            if n in run_parthenope.abundance_names]
data = {
    "small": [run_small.nuclear.Y_final[n] for n in nuclides],
    "small_parthenope": [run_parthenope.nuclear.Y_final[n] for n in nuclides],
}
df = pd.DataFrame(data, index=nuclides)
df["rel_diff"] = (df["small_parthenope"] - df["small"]) / df["small"]

print("Comparison of final abundances:")
print(df)

In [ ]:
# Plot relative difference of nuclide time evolution
plt.figure(figsize=(10, 6))

# Use the Y_of_t interpolant's time grid and the public T_of_t API
t_small = run_small.nuclear.Y_of_t.x
t_parthenope = run_parthenope.nuclear.Y_of_t.x
# T_of_t returns MeV; convert to Kelvin for the x-axis label
T_small = run_small.T_of_t(t_small) * run_small.cfg.MeV_to_Kelvin
T_parthenope = run_parthenope.T_of_t(t_parthenope) * run_parthenope.cfg.MeV_to_Kelvin

for n in nuclides:
    if n in run_small.abundance_names and n in run_parthenope.abundance_names:
        Y_small_fn = run_small[n]
        Y_parthenope_fn = run_parthenope[n]

        Y_s = Y_small_fn(t_small)
        Y_p = Y_parthenope_fn(t_parthenope)

        # Interpolate parthenope evolution onto small grid (T decreases with time)
        Y_p_interp = np.interp(T_small, T_parthenope[::-1], Y_p[::-1])
        rel_diff = (Y_p_interp - Y_s) / Y_s

        mask = Y_s > 1e-25
        plt.plot(T_small[mask], rel_diff[mask], label=n)

plt.xscale("log")
plt.gca().invert_xaxis()
plt.xlabel("Photon Temperature (K)")
plt.ylabel("Relative Difference (Parthenope - Small) / Small")
plt.legend()
plt.grid(True)
plt.show()